In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/28 01:21:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [65]:
df_sorted = (
    spark.sql("select * from serving_db.klines")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

In [66]:
df_sorted.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.

In [67]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("rsi6", types.DoubleType(), True)
])

In [68]:
from decimal import Decimal, getcontext, ROUND_HALF_UP

# set precision high enough for finance data
getcontext().prec = 28  

def rsi_in_chunks(iterator):  # one stream iterator per partition
    period_int = 6
    period = Decimal(period_int)
    ag, al = Decimal(0), Decimal(0)
    i = 0
    initialized = False
    buffer = []
    price_prev = None
    for pdf in iterator:  # 10,000 rows pandas dataframe for a chunk
        rsi = []
        for price in pdf["close_price"]:
            if price_prev is None:
                diff = None
            else:
                diff = price - price_prev
            price_prev = price
            if diff is None:
                rsi.append(None)
                continue
            
            if not initialized:
                buffer.append(diff)
                if len(buffer) < period_int:                   
                    rsi.append(None)
                    continue
                ag = sum(Decimal(diff) for diff in buffer if diff > 0) / period
                al = sum(Decimal(-diff) for diff in buffer if diff < 0) / period
                initialized = True
            else:
                ag = ((ag * (period - 1)) + (Decimal(diff) if diff > 0 else Decimal(0))) / period
                al = ((al * (period - 1)) + (Decimal(-diff) if diff < 0 else Decimal(0))) / period
            
            if al == 0:
                curr = Decimal(100)
            elif ag == 0:
                curr = Decimal(0)
            else:
                rs = ag / al
                curr = Decimal(100) - (Decimal(100) / (Decimal(1) + rs))
            # emulate Spark's round(..., 2)
            rsi.append(float(curr.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)))
        pdf["rsi6"] = rsi
        pdf = pdf[[*pdf.columns[:-1], "rsi6"]]
        yield pdf

In [71]:
rsi_df = df_sorted.mapInPandas(rsi_in_chunks, schema)

In [72]:
rsi_df.writeTo("serving_db.rsi6").tableProperty("format-version", "2").createOrReplace()

In [73]:
spark.sql("""
select *
from serving_db.rsi6
""").show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+-----+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time| rsi6|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+-----+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573| NULL|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993| NULL|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832| NULL|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074| NULL|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166| NULL|
| 194890